# 💼 NFL Transactions - Bronze Layer Ingestion

## 🎯 Purpose

Ingest official NFL transactions (signings, IR moves, activations, releases) into `main.fantasai.bronze_nfl_transactions`.

---

## 🔗 Data Source

**Source:** ESPN Transactions API  
**Endpoint:** `https://site.api.espn.com/apis/site/v2/sports/football/nfl/transactions`  
**Cost:** ✅ FREE (no API key required)  
**Rate Limits:** ✅ None observed  
**Coverage:** Official NFL transactions from all 32 teams  

---

## 📊 Transaction Types

### High Fantasy Value:
* **🏅 Signed** - New player signings (waiver opportunities)
* **🚫 Waived/Released** - Players available on waivers
* **🏯 Traded** - Opportunity changes
* **🏥 IR - Reserve/Injured** - Long-term injuries
* **✅ Activated** - Players returning from IR/suspension
* **🔼 Practice Squad** - Potential call-ups

### Medium Fantasy Value:
* **🚨 Suspended** - Availability changes
* **📄 Contract Extension** - Long-term security
* **⏸️ Reserve List** - Various statuses

---

## 📋 Data Fields

* `transaction_id` - Unique transaction ID
* `transaction_date` - Date of transaction
* `transaction_type` - Type (Signed, Waived, IR, etc.)
* `player_name` - Player name
* `position` - Player position
* `team` - Team (from or to)
* `description` - Full transaction description
* `espn_player_id` - ESPN player ID (if available)
* `fetched_at` - Ingestion timestamp

---

## ⚙️ Execution

**Schedule:** Daily at 8:00 AM UTC  
**Runtime:** ~30 seconds  
**Lookback:** Last 30 days of transactions  
**Deduplication:** Skips existing transactions  

---

## 💡 Fantasy Use Cases

1. **Waiver Wire Alerts** - New signings = pickup opportunities
2. **Injury Tracking** - IR moves = season-ending injuries
3. **Depth Chart Changes** - Releases = more opportunities for teammates
4. **Trade Impact** - New team = new offensive system
5. **Practice Squad Promotions** - Emergency call-ups before game day

---

## 📝 Notes

* Transactions posted within hours of official announcements
* Some transactions lack player IDs (undrafted, practice squad)
* Multiple transaction types can occur same day for one player
* Historical data available via date range parameters

In [0]:
# =============================================================================
# NFL TRANSACTIONS INGESTION - CONFIGURATION
# =============================================================================

from datetime import datetime, timedelta
import time

print("="*80)
print("💼 NFL Transactions Ingestion Configuration")
print("="*80)

# === API CONFIGURATION ===
ESPN_TRANSACTIONS_URL = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/transactions"
REQUEST_TIMEOUT = 15  # seconds

# === TABLE CONFIGURATION ===
BRONZE_TABLE = "main.fantasai.bronze_nfl_transactions"

# === LOOKBACK PERIOD ===
# Fetch transactions from last N days
LOOKBACK_DAYS = 30

print(f"\n📋 Configuration:")
print(f"   API Endpoint: {ESPN_TRANSACTIONS_URL}")
print(f"   Bronze Table: {BRONZE_TABLE}")
print(f"   Lookback Period: {LOOKBACK_DAYS} days")
print(f"   Request Timeout: {REQUEST_TIMEOUT}s")

print("\n" + "="*80)

In [0]:
# Install requests library
%pip install requests --quiet

print("✅ Dependencies installed")

In [0]:
# =============================================================================
# FETCH NFL TRANSACTIONS FROM ESPN API
# =============================================================================

import requests
import json
import re
import hashlib
import time
from datetime import datetime, timedelta, timezone
from typing import List, Dict, Optional
from collections import Counter

print("="*80)
print("📡 Fetching NFL Transactions")
print("="*80)

# Calculate date range (timezone-aware for comparison with API timestamps)
end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

print(f"\n📅 Date Range: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"\n⌛ Fetching transactions...\n")

# === FETCH FUNCTION ===
def fetch_nfl_transactions(timeout: int = REQUEST_TIMEOUT) -> Optional[Dict]:
    """Fetch NFL transactions from ESPN API."""
    try:
        response = requests.get(ESPN_TRANSACTIONS_URL, timeout=timeout)
        if response.status_code == 200:
            return response.json()
        else:
            return None
    except Exception as e:
        print(f"❌ Error fetching transactions: {str(e)}")
        return None

# Fetch transactions
data = fetch_nfl_transactions()

if not data or 'transactions' not in data:
    print("❌ No transaction data returned from API")
    all_transactions = []
else:
    transactions = data.get('transactions', [])
    total_count = data.get('count', 0)
    page_count = data.get('pageCount', 1)
    
    print(f"✅ Fetched page 1 of {page_count} ({len(transactions)} of {total_count} total transactions)")
    
    # === PAGINATION: Fetch all remaining pages ===
    if page_count > 1:
        print(f"\n📄 Fetching {page_count - 1} additional pages...\n")
        
        for page in range(2, page_count + 1):
            try:
                page_url = f"{ESPN_TRANSACTIONS_URL}?page={page}"
                page_response = requests.get(page_url, timeout=REQUEST_TIMEOUT)
                
                if page_response.status_code == 200:
                    page_data = page_response.json()
                    page_transactions = page_data.get('transactions', [])
                    transactions.extend(page_transactions)
                    
                    if page % 5 == 0:  # Progress every 5 pages
                        print(f"   ✓ Fetched pages 1-{page} ({len(transactions)} transactions so far)")
                else:
                    print(f"   ⚠️  Page {page} returned status {page_response.status_code}")
                    
                time.sleep(0.1)  # Rate limiting
                
            except Exception as e:
                print(f"   ⚠️  Error fetching page {page}: {str(e)}")
                continue
        
        print(f"\n✅ Fetched all {page_count} pages: {len(transactions)} total transactions")
    
    print(f"\n✅ Total fetched from API: {len(transactions)} transactions")
    
    # === PARSE TRANSACTIONS (NEW ESPN SCHEMA) ===
    all_transactions = []
    
    # Transaction type keywords (order matters - more specific first)
    TRANSACTION_TYPES = [
        ('Reserve/Injured', r'(?i)Reserve[/\s-]*Injured|placed.+on.+IR|Injured Reserve'),
        ('Reserve/COVID-19', r'(?i)Reserve[/\s-]*COVID'),
        ('Reserve/PUP', r'(?i)Reserve[/\s-]*PUP|Physically Unable'),
        ('Reserve/NFI', r'(?i)Reserve[/\s-]*NFI|Non-Football Injury'),
        ('Reserve/Suspended', r'(?i)Reserve[/\s-]*Suspended|placed.+on.+suspension'),
        ('Activated', r'(?i)Activated|activated from'),
        ('Waived', r'(?i)Waived|waived/'),
        ('Released', r'(?i)Released'),
        ('Signed', r'(?i)Signed|re-signed|signs|signed to'),
        ('Traded', r'(?i)Traded|trade'),
        ('Practice Squad', r'(?i)Practice Squad|practice-squad'),
        ('Designated to Return', r'(?i)Designated to Return|designated for return'),
        ('Terminated', r'(?i)Terminated'),
        ('Claimed', r'(?i)Claimed off waivers|claimed from'),
        ('Exempt', r'(?i)Exempt List'),
        ('Suspended', r'(?i)Suspended'),
        ('Contract Extension', r'(?i)Contract Extension|extension'),
    ]
    
    # Position patterns in descriptions
    POS_PATTERN = r'\b(QB|RB|WR|TE|OL|OT|OG|C|T/G|G/C|DE|DT|NT|LB|ILB|OLB|CB|S|SS|FS|K|P|LS|FB|DL|DB)\b'
    
    for txn in transactions:
        try:
            # Extract basic fields from NEW schema
            transaction_date_str = txn.get('date', '')
            description = txn.get('description', '')
            team_obj = txn.get('team', {})
            
            # Parse date
            transaction_date = None
            if transaction_date_str:
                try:
                    transaction_date = datetime.fromisoformat(transaction_date_str.replace('Z', '+00:00'))
                except:
                    pass
            
            # Filter by date range
            if transaction_date and transaction_date < start_date:
                continue
            
            # Generate transaction ID from hash (since API no longer provides ID)
            txn_hash = hashlib.md5(f"{transaction_date_str}_{description}".encode()).hexdigest()[:16]
            transaction_id = f"espn_{txn_hash}"
            
            # Extract team abbreviation
            team_code = team_obj.get('abbreviation', '') if team_obj else None
            
            # Extract transaction type from description
            transaction_type = None
            for txn_type, pattern in TRANSACTION_TYPES:
                if re.search(pattern, description):
                    transaction_type = txn_type
                    break
            
            # Extract player name: "[Action] [Position] [Name] to/from/off/etc"
            player_name = None
            name_pattern = r'(?:Signed|Waived|Released|Activated|Placed|Traded|Claimed|Designated)\s+(?:' + POS_PATTERN + r')\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+(?:\.)?)*?)\s+(?:to|from|off|on|and|for)'
            name_match = re.search(name_pattern, description, re.IGNORECASE)
            if name_match:
                player_name = name_match.group(2).strip()
            
            # Extract position from description
            position = None
            pos_match = re.search(POS_PATTERN, description)
            if pos_match:
                position = pos_match.group(1)
            
            # Build transaction record
            transaction_record = {
                'transaction_id': transaction_id,
                'transaction_date': transaction_date.isoformat() if transaction_date else None,
                'transaction_type': transaction_type,
                'player_name': player_name,
                'position': position,
                'team': team_code,
                'description': description,
                'espn_player_id': None,  # No longer available in new API
                'fetched_at': datetime.now(timezone.utc).isoformat()
            }
            
            all_transactions.append(transaction_record)
                
        except Exception as e:
            print(f"⚠️  Error parsing transaction: {str(e)}")
            continue
    
    print(f"\n📊 Parse Results:")
    print(f"   Transactions in Date Range: {len(all_transactions)}")
    
    # Show transaction type breakdown
    if all_transactions:
        type_counts = Counter([t['transaction_type'] for t in all_transactions])
        print(f"\n📋 Transaction Types:")
        for txn_type, count in sorted(type_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"   {txn_type}: {count}")
    
    print(f"\n✅ Successfully parsed {len(all_transactions)} transactions")

In [0]:
# =============================================================================
# DEBUG: Inspect Raw API Response
# =============================================================================

import requests
import json

print("="*80)
print("🔍 DEBUG: Inspecting ESPN Transactions API")
print("="*80)

url = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/transactions"

print(f"\n📡 Making request to: {url}\n")

try:
    response = requests.get(url, timeout=15)
    
    print(f"✅ HTTP Status Code: {response.status_code}")
    print(f"📊 Response Size: {len(response.content)} bytes")
    print(f"📝 Content-Type: {response.headers.get('Content-Type', 'unknown')}\n")
    
    if response.status_code == 200:
        try:
            data = response.json()
            
            # Show top-level keys
            print(f"📋 Top-Level Keys: {list(data.keys())}\n")
            
            # Show first 500 chars of raw response
            print("📄 First 500 chars of response:")
            print("="*80)
            print(response.text[:500])
            print("="*80)
            
            # Check for common transaction key names
            possible_keys = ['items', 'transactions', 'results', 'data', 'entries']
            print(f"\n🔎 Checking for transaction data in common keys...\n")
            
            for key in possible_keys:
                if key in data:
                    value = data[key]
                    if isinstance(value, list):
                        print(f"   ✅ Found '{key}': {len(value)} items")
                        if len(value) > 0:
                            print(f"      Sample item keys: {list(value[0].keys()) if isinstance(value[0], dict) else 'not a dict'}")
                    else:
                        print(f"   ⚠️  Found '{key}' but it's not a list: {type(value)}")
                else:
                    print(f"   ❌ '{key}' not found")
            
            # Check for pagination metadata
            if 'page' in data or 'pageInfo' in data or 'pagination' in data:
                print(f"\n📄 Pagination detected!")
                print(f"   page: {data.get('page', 'N/A')}")
                print(f"   pageInfo: {data.get('pageInfo', 'N/A')}")
                print(f"   pagination: {data.get('pagination', 'N/A')}")
                
        except json.JSONDecodeError as e:
            print(f"❌ JSON Decode Error: {str(e)}")
            print(f"\nRaw response (first 1000 chars):\n{response.text[:1000]}")
    else:
        print(f"❌ API returned status {response.status_code}")
        print(f"\nResponse body (first 500 chars):\n{response.text[:500]}")
        
except Exception as e:
    print(f"❌ Request Error: {str(e)}")
    import traceback
    print(f"\nFull traceback:\n{traceback.format_exc()}")

print("\n" + "="*80)

In [0]:
# =============================================================================
# DEBUG: Inspect Parsed Transaction Structure
# =============================================================================

import json

print("="*80)
print("🔍 DEBUG: Transaction Structure")
print("="*80)

# First show raw API structure
if transactions:
    print("\n🔍 Raw API Transaction Structure (first transaction):")
    print("="*80)
    print(json.dumps(transactions[0], indent=2))
    print("="*80)
    
if len(all_transactions) > 0:
    print(f"\n📊 Parsed {len(all_transactions)} transactions\n")
    print("📄 Sample transaction (first one):")
    print("="*80)
    print(json.dumps(all_transactions[0], indent=2))
    print("="*80)
    
    print(f"\n📄 Sample transaction (last one):")
    print("="*80)
    print(json.dumps(all_transactions[-1], indent=2))
    print("="*80)
else:
    print("\n❌ No transactions parsed!\n")
    print("🔍 Let's check the raw API transaction structure:\n")
    if transactions:
        print("Sample raw transaction from API:")
        print("="*80)
        print(json.dumps(transactions[0], indent=2)[:1000])
        print("="*80)

print("\n" + "="*80)

In [0]:
%sql
-- Create bronze table for NFL transactions (if not exists)

CREATE TABLE IF NOT EXISTS main.fantasai.bronze_nfl_transactions (
  transaction_id STRING NOT NULL COMMENT 'Unique transaction ID',
  transaction_date TIMESTAMP COMMENT 'Date of transaction',
  transaction_type STRING COMMENT 'Type (Signed, Waived, IR, etc.)',
  player_name STRING COMMENT 'Player name',
  position STRING COMMENT 'Player position',
  team STRING COMMENT 'Team (from or to)',
  description STRING COMMENT 'Full transaction description',
  espn_player_id STRING COMMENT 'ESPN player ID (if available)',
  fetched_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp',
  CONSTRAINT pk_nfl_transactions PRIMARY KEY (transaction_id, player_name)
)
COMMENT 'Official NFL transactions from ESPN API - Bronze layer'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

DESCRIBE EXTENDED main.fantasai.bronze_nfl_transactions;

In [0]:
# =============================================================================
# WRITE TO BRONZE TABLE WITH DEDUPLICATION
# =============================================================================

from pyspark.sql import functions as F

print("="*80)
print("💾 Writing Transactions to Bronze Table")
print("="*80)

if len(all_transactions) == 0:
    print("\n⚠️  No transactions to write. Skipping write operation.")
else:
    # Define explicit schema for Spark DataFrame
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType
    
    schema = StructType([
        StructField("transaction_id", StringType(), False),
        StructField("transaction_date", StringType(), True),
        StructField("transaction_type", StringType(), True),
        StructField("player_name", StringType(), True),
        StructField("position", StringType(), True),
        StructField("team", StringType(), True),
        StructField("description", StringType(), True),
        StructField("espn_player_id", StringType(), True),
        StructField("fetched_at", StringType(), False)
    ])
    
    # Convert to Spark DataFrame with explicit schema
    transactions_df = spark.createDataFrame(all_transactions, schema=schema)
    
    # Convert timestamp strings to proper timestamps
    transactions_df = transactions_df \
        .withColumn('transaction_date', F.to_timestamp('transaction_date')) \
        .withColumn('fetched_at', F.to_timestamp('fetched_at'))
    
    print(f"\n📊 Prepared {transactions_df.count()} transactions for insertion")
    
    # Check for existing transactions (deduplication)
    existing_transactions_df = spark.sql(f"""
        SELECT DISTINCT transaction_id, 
               COALESCE(player_name, 'TEAM_TXN') as player_name
        FROM {BRONZE_TABLE}
    """)
    
    existing_count = existing_transactions_df.count()
    print(f"📋 Found {existing_count} existing transactions in database")
    
    # Handle null player_names for join
    transactions_df = transactions_df.withColumn(
        'player_name_key',
        F.coalesce(F.col('player_name'), F.lit('TEAM_TXN'))
    )
    
    # Left anti join to find new transactions only
    new_transactions_df = transactions_df.join(
        existing_transactions_df,
        (transactions_df.transaction_id == existing_transactions_df.transaction_id) &
        (transactions_df.player_name_key == existing_transactions_df.player_name),
        how='left_anti'
    ).drop('player_name_key')
    
    new_count = new_transactions_df.count()
    duplicate_count = transactions_df.count() - new_count
    
    print(f"\n🔍 Deduplication results:")
    print(f"   New transactions: {new_count}")
    print(f"   Duplicates skipped: {duplicate_count}")
    
    if new_count > 0:
        print(f"\n💾 Writing {new_count} new transactions to {BRONZE_TABLE}...")
        
        new_transactions_df.write \
            .mode('append') \
            .saveAsTable(BRONZE_TABLE)
        
        print("\n✅ Write complete!")
        
        # Show sample of new transactions
        print("\n💼 Sample of new transactions:")
        new_transactions_df.select(
            'transaction_date',
            'transaction_type',
            'player_name',
            'position',
            'team',
            F.substring('description', 1, 60).alias('description_preview')
        ).orderBy(F.desc('transaction_date')).show(10, truncate=False)
    else:
        print("\n✓ No new transactions to write (all duplicates)")

print("\n" + "="*80)

In [0]:
%sql
-- Show most recent NFL transactions

SELECT 
  transaction_date,
  transaction_type,
  player_name,
  position,
  team,
  SUBSTRING(description, 1, 100) as description_preview,
  DATEDIFF(DAY, transaction_date, CURRENT_TIMESTAMP()) as days_ago
FROM main.fantasai.bronze_nfl_transactions
WHERE transaction_date IS NOT NULL
ORDER BY transaction_date DESC
LIMIT 25;

In [0]:
%sql
-- Summary statistics for NFL transactions table

SELECT 
  COUNT(*) as total_transactions,
  COUNT(DISTINCT player_name) as unique_players,
  COUNT(DISTINCT transaction_type) as transaction_types,
  COUNT(DISTINCT team) as teams_involved,
  MIN(transaction_date) as oldest_transaction,
  MAX(transaction_date) as newest_transaction,
  MAX(fetched_at) as last_ingestion_run,
  SUM(CASE WHEN espn_player_id IS NOT NULL THEN 1 ELSE 0 END) as transactions_with_player_ids
FROM main.fantasai.bronze_nfl_transactions;